In [1]:
# from ita_utils import *
# from common.legacy_utils import *
# from common.legacy_ita_utils import *

from common.trading_common import FrontOrderType, Exchange, CashMargin, MarginTradeType, TradingSide
from common.kabus_api import *

from concurrent.futures import ThreadPoolExecutor, as_completed

import websocket
import threading
import json
import time

current_date = datetime.now()
formatted_date = current_date.strftime("%Y%m%d")

# token = init_trading_api(refresh_token=False)
token = init_trading_api_from_general_config_server(False, False)

# test api init
# query_orders_after(f"{formatted_date}000000")

2026-01-07 18:14:30.733 | INFO     | common.trading_common:get_general_config:150 - get_general_config: kabus_token {'token': 'eeb1e5923c27456eb4b248bfc4082ad8', 'timestamp': '2026-01-07T18:14:21.044698', 'expires_at': '2026-01-08T18:14:21.044698'}
2026-01-07 18:14:30.735 | INFO     | common.kabus_api:init_trading_api_from_general_config_server:97 - No token found in general config server, will request new token from api
2026-01-07 18:14:30.735 | INFO     | common.kabus_api:init_trading_api:145 - Requesting new token from API
2026-01-07 18:14:30.742 | INFO     | common.kabus_api:init_trading_api:147 - init_trading_api: {"ResultCode":0,"Token":"c0b56519efe94f75b0217828dd3d8200"}
2026-01-07 18:14:30.744 | INFO     | common.kabus_api:init_trading_api:164 - Token saved to C:\Users\brisk2\Documents\GitHub\brisk-hack\gomihiroi\common\kabus_token.json
2026-01-07 18:14:30.761 | INFO     | common.trading_common:put_general_config:156 - put_general_config: kabus_token ['ok']


In [2]:
from loguru import logger
logger.remove()

_lock = threading.Lock()
_symbol_sink_id = {}  # symbol -> sink_id
sc_name = {}

@logger.catch
@require_token
def register_fut(fut, exchange):
    req = requests.put(f'{kabucom_endpoint}/register', json={'Symbols': [{'Symbol': fut, 'Exchange': exchange}]}, headers={'Host': host, 'X-API-KEY': f'{token}'}, timeout=timeout)
    logger.info(f'register_fut: {req.text}')
    res = req.json()
    logger.info(f'register_fut: {res}')

@logger.catch
@require_token
def get_symbol_fut(futureCode: str, derivMonth: int):
    req = requests.get(f'{kabucom_endpoint}/symbolname/future', params={'FutureCode': f'{futureCode}', 'DerivMonth': derivMonth}, headers={'Host': host, 'X-API-KEY': f'{token}'}, timeout=timeout)
    logger.info(f'get_symbol_fut: {req.text}')
    res = req.json()
    logger.info(f'get_symbol_fut: {res}')
    if res.get('Symbol', None) is not None:
        return res['Symbol']
    else:
        logger.error(f'Failed to get symbol for {futureCode} {derivMonth}, response: {res}')
        return None

q_month = 202603
r_month = 202601

fut_symbol = [
    {'name': 'TOPIXmini', 'month': q_month},
    {'name': 'NK225mini', 'month': q_month},
    {'name': 'NK225mini', 'month': r_month},
    {'name': 'NK225', 'month': q_month},
    {'name': 'NK225micro', 'month': q_month},
    {'name': 'TOPIX', 'month': q_month},
    {'name': 'GROWTH', 'month': q_month},
    {'name': 'REIT', 'month': q_month},
]

unregister_all_sc()

for fut in fut_symbol:
    fut['symbol'] = get_symbol_fut(fut['name'], fut['month'])
    time.sleep(0.5)
    sc_name[fut['symbol']] = f'{fut["name"]}_{fut["month"]}_{fut["symbol"]}'
    print(fut)
    # 2: all day 23: day 24: night
    register_fut(fut['symbol'], 2)


def ensure_symbol_sink(symbol: str) -> int:
    """确保该 symbol 的文件 sink 已创建，返回 sink_id。线程安全。"""
    sink_id = _symbol_sink_id.get(symbol)
    if sink_id is not None:
        return sink_id

    with _lock:
        sink_id = _symbol_sink_id.get(symbol)
        if sink_id is not None:
            return sink_id

        # 每个 symbol 一个文件（JSONL）
        path = f"snapshots/orderbook_{sc_name[symbol]}.jsonl"

        sink_id = logger.add(
            path,
            # rotation="512 MB"
            rotation="08:00",
            retention=10,
            format="{message}",
            enqueue=True,          # ⭐ 必须：避免阻塞 on_message 线程
            backtrace=False,
            diagnose=False,
            filter=lambda record, s=symbol: record["extra"].get("symbol") == s,
        )
        _symbol_sink_id[symbol] = sink_id
        return sink_id

{'name': 'TOPIXmini', 'month': 202603, 'symbol': '161030006'}
{'name': 'NK225mini', 'month': 202603, 'symbol': '161030019'}
{'name': 'NK225mini', 'month': 202601, 'symbol': '161010019'}
{'name': 'NK225', 'month': 202603, 'symbol': '161030018'}
{'name': 'NK225micro', 'month': 202603, 'symbol': '161030023'}
{'name': 'TOPIX', 'month': 202603, 'symbol': '161030005'}
{'name': 'GROWTH', 'month': 202603, 'symbol': '161030011'}
{'name': 'REIT', 'month': 202603, 'symbol': '161030069'}


In [3]:
# compress snapshot
TOP_FIELD_MAP = {
    "Symbol": "s",
    "SymbolName": "sn",
    "Exchange": "ex",
    "ExchangeName": "exn",
    "SecurityType": "st",

    "currentTime": "ct",                 # 你本地采集时间
    "CurrentPrice": "px",
    "CurrentPriceTime": "pxt",
    "CurrentPriceChangeStatus": "pxcs",
    "CurrentPriceStatus": "pxst",
    "CalcPrice": "cpx",

    "PreviousClose": "pc",
    "PreviousCloseTime": "pct",
    "ChangePreviousClose": "chg",
    "ChangePreviousClosePer": "chgp",

    "OpeningPrice": "op",
    "OpeningPriceTime": "opt",
    "HighPrice": "hp",
    "HighPriceTime": "hpt",
    "LowPrice": "lp",
    "LowPriceTime": "lpt",

    "TradingVolume": "v",
    "TradingVolumeTime": "vt",
    "VWAP": "vwap",
    "TradingValue": "tv",

    "ClearingPrice": "clr",              # 你的样例里有
}

DROP_REDUNDANT = {
    "BidQty", "BidPrice", "BidSign",
    "AskQty", "AskPrice", "AskSign",
}

def compress_orderbook_snapshot(raw: dict) -> dict:
    out = {}

    # 1) 顶层字段 remap（并跳过冗余）
    for k, short in TOP_FIELD_MAP.items():
        if k in raw and raw[k] is not None:
            out[short] = raw[k]

    # 2) 压缩 Sell/Buy 10档
    def pack_side(prefix: str) -> list:
        levels = []
        for i in range(1, 11):
            lv = raw.get(f"{prefix}{i}")
            if not isinstance(lv, dict):
                levels.append(None)
                continue

            p = lv.get("Price")
            q = lv.get("Qty")
            if p is None and q is None:
                levels.append(None)
                continue

            if i == 1:
                # level1 可能有 Sign
                levels.append([p, q, lv.get("Sign")])
            else:
                levels.append([p, q])
        return levels

    out["S"] = pack_side("Sell")
    out["B"] = pack_side("Buy")

    return out


In [4]:
# init ws for push api

def on_message(ws, message):
    m = json.loads(message)
    # 0 for index, 1 for stock, different FUT has different kind of values
    if m.get('SecurityType') in [0, 1]:
        return
    
    m['currentTime'] = datetime.now().isoformat()
    symbol = m.get('Symbol')
    if symbol is None:
        return

    ensure_symbol_sink(symbol)

    logger.bind(symbol=symbol).info(
        json.dumps(compress_orderbook_snapshot(m), ensure_ascii=False, separators=(",", ":"))
    )

def on_error(ws, error):
    print('--- ERROR --- ')
    print(error)

def on_close(ws, arg2, arg3):
    print('--- DISCONNECTED --- ')

def on_open(ws):
    print('--- CONNECTED --- ')
    
def run_ws():
    global ws_global
    url = 'ws://192.168.50.131:16080/kabusapi/websocket'
    # original upstream
    # url = 'ws://192.168.50.131:17080/kabusapi/websocket'
    ws = websocket.WebSocketApp(
        url,
        on_open=on_open,
        on_message=on_message,
        on_error=on_error,
        on_close=on_close
    )
    ws_global = ws
    while not stop_event.is_set():
        ws.run_forever()
        time.sleep(1)  # 防止重连时过于频繁


# kabus websocket
tick_lock = threading.Lock()
ws_global = None
stop_event = threading.Event()

ws_thread = threading.Thread(target=run_ws, daemon=True)
ws_thread.start()

--- CONNECTED --- 
